# Pandas Lesson 2: Filtering and Ordering Data

In this lesson, you will learn how to **keep only the rows and columns you care about** (filtering) and how to **sort your data** (ordering) with pandas.

These are two of the things you will do most often when working with data, so it is worth taking the time to understand them well.

### What you will learn

1. Filtering rows based on the values in a column
2. Filtering using the index (`filter`, `loc`, `iloc`)
3. Sorting data with `sort_values`

> This notebook follows the pandas tutorial series by Alex The Analyst, with my own notes and explanations.

## 0. Setup

### The dataset

We are using `world_population.csv`, a dataset with one row per country. It includes the country's rank, its 3-letter code (`CCA3`), capital, continent, population for several years, area, density, growth rate and share of the world population.

> **Tip:** put `world_population.csv` in the same folder as this notebook so the code below works without changing the file path.

### Importing pandas

We import pandas and give it the short name `pd`. This is the standard convention: almost everyone writes it this way.

In [ ]:
import pandas as pd

### Loading the data

`pd.read_csv()` reads the CSV file and returns a **DataFrame** (a table with rows and columns). We store it in a variable called `df` so we can reuse it.

In [ ]:
df = pd.read_csv('world_population.csv')
df

`df.head()` shows only the first 5 rows. It is a quick way to check what the data looks like without printing everything.

In [ ]:
df.head()

---
## 1. Filtering Rows Based on Column Values

The idea is simple: we look at a column, apply a condition, and keep only the rows where the condition is true.

### 1.1 Comparison operators

First, let's see what a condition actually returns on its own.

In [ ]:
df['Rank'] <= 10

This does **not** give us rows. It gives a list of `True` / `False` values, one per row. This is called a **boolean mask**.

To actually keep the rows, we put that mask inside `df[...]`. Pandas then keeps only the rows where the mask is `True`.

In [ ]:
# Keep only the countries ranked in the top 10
df[df['Rank'] <= 10]

You can use any comparison operator:

| Operator | Meaning |
|---|---|
| `>` | greater than |
| `<` | less than |
| `>=` | greater than or equal to |
| `<=` | less than or equal to |
| `==` | equal to |
| `!=` | not equal to |

Notice the difference: `< 10` gives ranks 1 to 9, while `<= 10` also includes rank 10.

In [ ]:
df[df['Rank'] < 10]

### 1.2 Filtering on a list of values with `isin()`

What if we want a few specific countries? We put them in a list, then use `.isin()` to keep the rows whose value is **in** that list.

If you know SQL, this works like `WHERE country IN (...)`.

In [ ]:
specific_countries = ['Bangladesh', 'Brazil']

df[df['Country'].isin(specific_countries)]

The same result can be written with `==` and the `|` symbol, which means **OR**.

Each condition must be wrapped in its own parentheses. With only two countries this is fine, but with a long list, `isin()` is much cleaner.

In [ ]:
df[(df['Country'] == 'Bangladesh') | (df['Country'] == 'Brazil')]

### 1.3 Combining conditions with `&` (AND)

`|` means OR, and `&` means **AND**: both conditions must be true.

Here we keep the countries that are in Asia **and** in the top 20.

In [ ]:
df[(df['Continent'] == 'Asia') & (df['Rank'] <= 20)]

### 1.4 Searching for text with `str.contains()`

`isin()` needs the exact value. Sometimes we only know **part** of the text.

`.str.contains()` keeps the rows where the text contains a given word. If you know SQL, this is similar to `LIKE '%United%'`.

In [ ]:
# Keep countries whose name contains the word "United"
df[df['Country'].str.contains('United')]

---
## 2. Filtering Using the Index

So far we filtered on columns. We can also filter on the **index**, which is the label on the left of each row.

### 2.1 Setting a better index

Right now the index is just `0, 1, 2, ...`, which is not very meaningful. Each country appears only once, so `Country` makes a great index.

`set_index('Country')` turns the `Country` column into the index. We save the result in a new variable, `df2`, so the original `df` stays unchanged and we can switch between them.

In [ ]:
df2 = df.set_index('Country')
df2

### 2.2 The `filter()` method

`filter()` lets us choose which **columns** or which **rows** to keep, based on their labels.

The `axis` parameter tells pandas where to look:

| `axis` | Looks at... |
|---|---|
| `axis=1` | the columns (top labels), this is the default |
| `axis=0` | the rows (index labels on the left) |

**Keeping specific columns** with `items`:

In [ ]:
df2.filter(items=['Continent', 'CCA3'])

This is the same thing, but written with `axis=1` explicitly:

In [ ]:
df2.filter(items=['Continent', 'CCA3'], axis=1)

**Keeping specific rows** with `axis=0`: here we search the index for `Zambia`.

In [ ]:
df2.filter(items=['Zambia'], axis=0)

**Partial match** with `like`: instead of an exact label, `like` keeps every label that *contains* the text.

This gives the same result as `str.contains()` earlier, but it works on the index.

In [ ]:
df2.filter(like='United', axis=0)

### 2.3 `loc` and `iloc`

These are the two most common ways to select rows by index.

| Method | Stands for | Selects by... |
|---|---|---|
| `loc` | location | the **label** (the name, e.g. `'United States'`) |
| `iloc` | integer location | the **position** (the number, e.g. `3`) |

Even when the index is made of country names, every row still has a hidden position number starting at 0. That is what `iloc` uses.

**`loc`** finds the row by its name:

In [ ]:
df2.loc['United States']

**`iloc`** finds the row by its position. Position `3` is the 4th row, because counting starts at 0.

In [ ]:
df2.iloc[3]

---
## 3. Sorting Data with `sort_values()`

Filtered data is often still in a random order. `sort_values()` puts it in the order we want.

### 3.1 Sorting by one column

The `by` parameter is the column to sort on. By default, sorting is **ascending** (smallest to largest).

In [ ]:
df[df['Rank'] < 10].sort_values(by='Rank')

Writing `ascending=True` gives the exact same result, since it is the default:

In [ ]:
df[df['Rank'] < 10].sort_values(by='Rank', ascending=True)

To sort from **largest to smallest** (descending), use `ascending=False`:

In [ ]:
df[df['Rank'] < 10].sort_values(by='Rank', ascending=False)

### 3.2 Sorting by several columns

We can pass a **list** of columns. The order of the list matters:

1. Pandas sorts by the **first** column.
2. When two rows have the same value in the first column, it uses the **second** column to break the tie.

Here we sort by `Continent` first, then by `Country` inside each continent.

In [ ]:
df[df['Rank'] < 10].sort_values(by=['Continent', 'Country'])

### 3.3 A different direction for each column

`ascending` can also be a **list**, with one value per column, in the same order.

Below: `Continent` is sorted descending (Z to A), and `Country` is sorted ascending (A to Z) within each continent. Look at Asia to see the countries in alphabetical order.

> **Watch out:** writing `ascending=False, True` (without brackets) causes an error. The two values must be inside a list: `ascending=[False, True]`.

In [ ]:
df[df['Rank'] < 10].sort_values(by=['Continent', 'Country'], ascending=[False, True])

---
## Summary

| Goal | Code |
|---|---|
| Filter with a condition | `df[df['Rank'] <= 10]` |
| Filter on a list of values | `df[df['Country'].isin(['Brazil', 'Bangladesh'])]` |
| Combine conditions (OR / AND) | `df[(cond1) \| (cond2)]` / `df[(cond1) & (cond2)]` |
| Search for text in a column | `df[df['Country'].str.contains('United')]` |
| Set a column as the index | `df2 = df.set_index('Country')` |
| Keep specific columns | `df2.filter(items=['Continent', 'CCA3'])` |
| Keep specific rows by label | `df2.filter(items=['Zambia'], axis=0)` |
| Search the index for text | `df2.filter(like='United', axis=0)` |
| Select a row by name | `df2.loc['United States']` |
| Select a row by position | `df2.iloc[3]` |
| Sort by one column | `df.sort_values(by='Rank', ascending=False)` |
| Sort by several columns | `df.sort_values(by=['Continent', 'Country'], ascending=[False, True])` |

**Next lesson:** Indexing in pandas.